In [ ]:
! pip install pytelegrambotapi pandas numpy matplotlib openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 6.0 MB/s eta 0:00:00


In [4]:
'''
Modules:
  - 🎮 Fun Work     : Calculator, Timer, Location, Sticker  (Shugyla)
  - 📊 Data Analytics: CSV/XLSX upload, stats, histogram     (Iskakova Zhanel)
  - 📦 ZIP Search   : ZIP upload, search by word/city/year   (Tursynbay Aspan)
'''

import telebot
from telebot import types
import threading
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TOKEN = "8597335745:AAGvMj5fQ0GcPdtJLuGaXZZAas3a0kzIsx8"
bot = telebot.TeleBot(TOKEN)

user_states      = {}
user_dataframes  = {}
user_zip_files   = {}

calc_expr        = {}
calc_just_result = {}
last_sticker     = {}
STICKERS = {
    "calculator": "",
    "timer":      "",
    "location":   "",
}

def get_main_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎮 Fun Work")
    markup.row("📊 Data Analytics", "📦 ZIP Search")
    return markup

def get_fun_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("🧮 Calculator", "⏱ Timer")
    markup.add("📍 Location",   "🎭 Sticker")
    markup.add("⬅️ Back")
    return markup

def get_calc_keyboard():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("7", "8", "9")
    markup.row("4", "5", "6")
    markup.row("1", "2", "3")
    markup.row(".", "0")
    markup.row("➕", "➖", "✖️", "➗")
    markup.row("=", "( )", "⌫", "🗑")
    markup.row("⬅️ Back")
    return markup

def get_timer_keyboard():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("⏱ 30 sec",  "⏱ 1 min")
    markup.add("⏱ 5 min",   "⏱ 10 min")
    markup.add("⏱ 15 min",  "⏱ 30 min")
    markup.add("⏱ 1 hour",  "⌨️ Custom time")
    markup.add("⬅️ Back")
    return markup

def get_sticker_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.add("🧮 Set for Calculator")
    markup.add("⏱ Set for Timer")
    markup.add("📍 Set for Location")
    markup.add("⬅️ Back")
    return markup

def get_analytics_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("Load Dataset", "Dataset Info")
    markup.row("Summary", "Missing Values")
    markup.row("Correlation", "Histogram")
    markup.row("Search Column", "Export Clean Data")
    markup.row("Help Analytics", "⬅️ Back")
    return markup

def get_zip_menu():
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("files", "summary")
    markup.row("search", "city")
    markup.row("year", "find")
    markup.row("help zip", "⬅️ Back")
    return markup

SYMBOL_MAP = {"➕": "+", "➖": "-", "✖️": "*", "➗": "/"}

def send_calc(chat_id, expr):
    display = expr if expr else "0"
    bot.send_message(chat_id, f"🧮 {display}", reply_markup=get_calc_keyboard())

def _start_timer(chat_id, seconds):
    if seconds >= 3600:
        label = f"{seconds//3600}h {(seconds%3600)//60}m" if seconds%3600 else f"{seconds//3600}h"
    elif seconds >= 60:
        label = f"{seconds//60}m {seconds%60}s" if seconds%60 else f"{seconds//60}m"
    else:
        label = f"{seconds}s"
    bot.send_message(chat_id, f"✅ Timer set for {label}. I'll notify you!", reply_markup=get_timer_keyboard())
    def notify():
        bot.send_message(chat_id, f"⏰ Time is up! Your {label} timer is done.")
    threading.Timer(seconds, notify).start()

def get_df(chat_id):
    return user_dataframes.get(chat_id, None)

def get_user_zip(chat_id):
    return user_zip_files.get(chat_id, None)

def is_text_file(file_name):
    return file_name.endswith(".txt") or file_name.endswith(".csv")

def read_file_from_zip(zip_file, file_name):
    try:
        content = zip_file.read(file_name)
        try:
            return content.decode("utf-8")
        except:
            return content.decode("cp1251")
    except:
        return ""

@bot.message_handler(commands=['start', 'menu'])
def start_message(message):
    user_states[message.chat.id] = None
    bot.send_message(
        message.chat.id,
        "👋 Welcome to the Team Bot!\n\n"
        "Choose a section:",
        reply_markup=get_main_menu()
    )

@bot.message_handler(content_types=['document'])
def handle_document(message):
    chat_id = message.chat.id
    state   = user_states.get(chat_id)
    file_name = message.document.file_name

    if file_name.endswith(".zip"):
        try:
            if not os.path.exists("downloads"):
                os.mkdir("downloads")
            file_info = bot.get_file(message.document.file_id)
            downloaded = bot.download_file(file_info.file_path)
            path = f"downloads/{chat_id}_{file_name}"
            with open(path, "wb") as f:
                f.write(downloaded)
            user_zip_files[chat_id] = path
            bot.send_message(
                chat_id,
                "✅ ZIP saved! Use the ZIP Search menu.",
                reply_markup=get_zip_menu()
            )
            user_states[chat_id] = "zip"
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error saving ZIP: {e}")
        return

    if file_name.endswith(".csv") or file_name.endswith(".xlsx"):
        try:
            file_info = bot.get_file(message.document.file_id)
            downloaded = bot.download_file(file_info.file_path)
            with open(file_name, "wb") as f:
                f.write(downloaded)
            df = pd.read_csv(file_name) if file_name.endswith(".csv") else pd.read_excel(file_name)
            user_dataframes[chat_id] = df
            bot.send_message(chat_id, "✅ Dataset loaded!")
            bot.send_message(chat_id, str(df.head()), reply_markup=get_analytics_menu())
            user_states[chat_id] = "analytics"
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}")
        return

    bot.send_message(chat_id, "⚠️ Please send a ZIP, CSV, or XLSX file.")

@bot.message_handler(content_types=['location'])
def handle_location(message):
    chat_id = message.chat.id
    lat = message.location.latitude
    lon = message.location.longitude
    maps_link = f"https://maps.google.com/?q={lat},{lon}"
    bot.send_message(
        chat_id,
        f"📍 Your location:\nLatitude:  {lat}\nLongitude: {lon}\n\n"
        f"Open in Google Maps: {maps_link}",
        reply_markup=get_fun_menu()
    )

@bot.message_handler(content_types=['sticker'])
def handle_sticker(message):
    chat_id = message.chat.id
    last_sticker[chat_id] = message.sticker.file_id
    state = user_states.get(chat_id)
    if state == "setsticker_calc":
        STICKERS["calculator"] = last_sticker[chat_id]
        bot.send_message(chat_id, "✅ Sticker saved for Calculator!", reply_markup=get_sticker_menu())
        user_states[chat_id] = "sticker_menu"
    elif state == "setsticker_timer":
        STICKERS["timer"] = last_sticker[chat_id]
        bot.send_message(chat_id, "✅ Sticker saved for Timer!", reply_markup=get_sticker_menu())
        user_states[chat_id] = "sticker_menu"
    elif state == "setsticker_loc":
        STICKERS["location"] = last_sticker[chat_id]
        bot.send_message(chat_id, "✅ Sticker saved for Location!", reply_markup=get_sticker_menu())
        user_states[chat_id] = "sticker_menu"
    else:
        bot.send_message(chat_id, "Sticker received! Use 🎭 Sticker menu to assign it.", reply_markup=get_fun_menu())

@bot.message_handler(content_types=['text'])
def handle_text(message):
    chat_id   = message.chat.id
    text      = message.text.strip()
    lower     = text.lower()
    state     = user_states.get(chat_id)

    if text == "⬅️ Back":
        calc_expr[chat_id] = ""
        user_states[chat_id] = None
        if state in ("calc", "timer", "timer_custom", "sticker_menu",
                     "setsticker_calc", "setsticker_timer", "setsticker_loc"):
            bot.send_message(chat_id, "🎮 Fun section:", reply_markup=get_fun_menu())
            user_states[chat_id] = "fun"
        elif state == "fun":
            bot.send_message(chat_id, "Main menu:", reply_markup=get_main_menu())
            user_states[chat_id] = None
        elif state == "analytics":
            bot.send_message(chat_id, "Main menu:", reply_markup=get_main_menu())
            user_states[chat_id] = None
        elif state == "zip":
            bot.send_message(chat_id, "Main menu:", reply_markup=get_main_menu())
            user_states[chat_id] = None
        else:
            bot.send_message(chat_id, "Main menu:", reply_markup=get_main_menu())
            user_states[chat_id] = None
        return

    if text == "🎮 Fun Work":
        user_states[chat_id] = "fun"
        bot.send_message(chat_id, "🎮 Fun section - choose a feature:", reply_markup=get_fun_menu())
        return

    if text == "📊 Data Analytics":
        user_states[chat_id] = "analytics"
        bot.send_message(
            chat_id,
            "📊 Data Analytics\n\nSend me a CSV or XLSX file to start.",
            reply_markup=get_analytics_menu()
        )
        return

    if text == "📦 ZIP Search":
        user_states[chat_id] = "zip"
        bot.send_message(
            chat_id,
            "📦 ZIP Search\n\nSend me a ZIP file to start.",
            reply_markup=get_zip_menu()
        )
        return

    if text == "🧮 Calculator":
        user_states[chat_id] = "calc"
        calc_expr[chat_id] = ""
        calc_just_result[chat_id] = False
        if STICKERS["calculator"]:
            bot.send_sticker(chat_id, STICKERS["calculator"])
        send_calc(chat_id, "")
        return

    if state == "calc":
        if text in SYMBOL_MAP:
            calc_just_result[chat_id] = False
            calc_expr[chat_id] += SYMBOL_MAP[text]
            send_calc(chat_id, calc_expr[chat_id])
            return
        if text in "0123456789.":
            if calc_just_result.get(chat_id):
                calc_expr[chat_id] = ""
                calc_just_result[chat_id] = False
            calc_expr[chat_id] += text
            send_calc(chat_id, calc_expr[chat_id])
            return
        if text == "( )":
            expr = calc_expr.get(chat_id, "")
            calc_expr[chat_id] += "(" if (not expr or expr[-1] in "+-*/(") else ")"
            send_calc(chat_id, calc_expr[chat_id])
            return
        if text == "⌫":
            calc_expr[chat_id] = calc_expr.get(chat_id, "")[:-1]
            send_calc(chat_id, calc_expr[chat_id])
            return
        if text == "🗑":
            calc_expr[chat_id] = ""
            send_calc(chat_id, "")
            return
        if text == "=":
            expr = calc_expr.get(chat_id, "")
            if not expr:
                bot.send_message(chat_id, "⚠️ Nothing to calculate.", reply_markup=get_calc_keyboard())
                return
            try:
                result = eval(expr)
                bot.send_message(chat_id, f"✅ {expr} = {result}", reply_markup=get_calc_keyboard())
                calc_expr[chat_id] = str(result)
                calc_just_result[chat_id] = True
            except ZeroDivisionError:
                bot.send_message(chat_id, "⚠️ Division by zero!", reply_markup=get_calc_keyboard())
                calc_expr[chat_id] = ""
            except Exception:
                bot.send_message(chat_id, "⚠️ Invalid expression.", reply_markup=get_calc_keyboard())
                calc_expr[chat_id] = ""
            return

    if text == "⏱ Timer":
        user_states[chat_id] = "timer"
        if STICKERS["timer"]:
            bot.send_sticker(chat_id, STICKERS["timer"])
        bot.send_message(chat_id, "⏱ Choose how long:", reply_markup=get_timer_keyboard())
        return

    TIMER_PRESETS = {
        "⏱ 30 sec": 30, "⏱ 1 min": 60, "⏱ 5 min": 300,
        "⏱ 10 min": 600, "⏱ 15 min": 900, "⏱ 30 min": 1800, "⏱ 1 hour": 3600,
    }
    if text in TIMER_PRESETS and state == "timer":
        _start_timer(chat_id, TIMER_PRESETS[text])
        return

    if text == "⌨️ Custom time" and state == "timer":
        user_states[chat_id] = "timer_custom"
        m = types.ReplyKeyboardMarkup(resize_keyboard=True)
        m.add("⬅️ Back")
        bot.send_message(chat_id, "⌨️ Type number of seconds (Example: 45):", reply_markup=m)
        return

    if state == "timer_custom":
        try:
            seconds = int(text)
            if seconds <= 0:
                bot.send_message(chat_id, "⚠️ Enter a positive number.")
                return
            if seconds > 86400:
                bot.send_message(chat_id, "⚠️ Maximum is 86400 seconds (24 hours).")
                return
            user_states[chat_id] = "timer"
            _start_timer(chat_id, seconds)
        except ValueError:
            bot.send_message(chat_id, "⚠️ Please type a number. Example: 45")
        return

    if text == "📍 Location":
        if STICKERS["location"]:
            bot.send_sticker(chat_id, STICKERS["location"])
        m = types.ReplyKeyboardMarkup(resize_keyboard=True, one_time_keyboard=True)
        m.add(types.KeyboardButton("📍 Send my location", request_location=True))
        m.add("⬅️ Back")
        bot.send_message(chat_id, "📍 Press the button to share your GPS:", reply_markup=m)
        return

    if text == "🎭 Sticker":
        user_states[chat_id] = "sticker_menu"
        bot.send_message(
            chat_id,
            "🎭 Sticker settings\nChoose a feature and then send a sticker.",
            reply_markup=get_sticker_menu()
        )
        return

    if text == "🧮 Set for Calculator":
        user_states[chat_id] = "setsticker_calc"
        bot.send_message(chat_id, "Send a sticker to attach to Calculator 🧮")
        return
    if text == "⏱ Set for Timer":
        user_states[chat_id] = "setsticker_timer"
        bot.send_message(chat_id, "Send a sticker to attach to Timer ⏱")
        return
    if text == "📍 Set for Location":
        user_states[chat_id] = "setsticker_loc"
        bot.send_message(chat_id, "Send a sticker to attach to Location 📍")
        return

    if text == "Load Dataset":
        user_states[chat_id] = "analytics"
        bot.send_message(chat_id, "📂 Send a CSV or XLSX file.", reply_markup=get_analytics_menu())
        return

    if text == "Dataset Info":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        info = f"Rows: {df.shape[0]}\nColumns: {df.shape[1]}\n\nColumn names:\n{list(df.columns)}"
        bot.send_message(chat_id, info, reply_markup=get_analytics_menu())
        return

    if text == "Summary":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        bot.send_message(chat_id, str(df.describe()), reply_markup=get_analytics_menu())
        return

    if text == "Missing Values":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        bot.send_message(chat_id, str(df.isnull().sum()), reply_markup=get_analytics_menu())
        return

    if text == "Correlation":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        numeric_df = df.select_dtypes(include=np.number)
        bot.send_message(chat_id, str(numeric_df.corr()), reply_markup=get_analytics_menu())
        return

    if text.startswith("Histogram"):
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        parts = text.split(maxsplit=1)
        if len(parts) < 2:
            columns = "\n".join(df.columns)
            bot.send_message(chat_id, f"Please specify column.\nExample: Histogram age\n\nAvailable:\n{columns}", reply_markup=get_analytics_menu())
            return
        column = parts[1]
        if column not in df.columns:
            bot.send_message(chat_id, f"Column '{column}' not found.\nAvailable:\n{chr(10).join(df.columns)}", reply_markup=get_analytics_menu())
            return
        if not np.issubdtype(df[column].dtype, np.number):
            bot.send_message(chat_id, "⚠️ Histogram works only with numeric columns.", reply_markup=get_analytics_menu())
            return
        try:
            plt.figure(figsize=(8, 5))
            df[column].hist()
            plt.title(f"Histogram of {column}")
            plt.xlabel(column)
            plt.ylabel("Frequency")
            fname = f"{column}_histogram.png"
            plt.savefig(fname)
            plt.close()
            with open(fname, "rb") as photo:
                bot.send_photo(chat_id, photo)
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_analytics_menu())
        return

    if text == "Search Column":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        columns = "\n".join(df.columns)
        bot.send_message(chat_id, f"Available columns:\n{columns}\n\nExample: Histogram age", reply_markup=get_analytics_menu())
        return

    if text == "Export Clean Data":
        df = get_df(chat_id)
        if df is None:
            bot.send_message(chat_id, "⚠️ Please upload a dataset first.", reply_markup=get_analytics_menu())
            return
        clean_df = df.dropna()
        clean_df.to_csv("clean_dataset.csv", index=False)
        with open("clean_dataset.csv", "rb") as f:
            bot.send_document(chat_id, f)
        return

    if text == "Help Analytics":
        help_text = (
            "📊 DATA ANALYTICS COMMANDS\n\n"
            "Load Dataset - send CSV/XLSX file\n"
            "Dataset Info - rows, columns\n"
            "Summary - statistics\n"
            "Missing Values - count of NaN\n"
            "Correlation - correlation matrix\n"
            "Histogram <column> - build chart\n"
            "Search Column - list columns\n"
            "Export Clean Data - download cleaned file"
        )
        bot.send_message(chat_id, help_text, reply_markup=get_analytics_menu())
        return

    if lower == "files":
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                files = zf.namelist()
            answer = "📂 Files inside ZIP\n━━━━━━━━━━━━━━━━━━━━\n\n"
            for i, fn in enumerate(files, 1):
                answer += f"{i}. 📄 {fn}\n"
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "summary":
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        try:
            info = {"files_count": 0, "txt_files": 0, "csv_files": 0, "total_lines": 0}
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fn in zf.namelist():
                    info["files_count"] += 1
                    if fn.endswith(".txt"): info["txt_files"] += 1
                    if fn.endswith(".csv"): info["csv_files"] += 1
                    if is_text_file(fn):
                        info["total_lines"] += len(read_file_from_zip(zf, fn).split("\n"))
            answer = (
                f"📊 Archive summary\n━━━━━━━━━━━━━━━━━━━━\n\n"
                f"📦 All files: {info['files_count']}\n"
                f"📝 TXT files: {info['txt_files']}\n"
                f"📑 CSV files: {info['csv_files']}\n"
                f"📌 Total lines: {info['total_lines']}"
            )
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "search":
        bot.send_message(chat_id, "🔎 Write word after command.\nExample: search Astana", reply_markup=get_zip_menu())
        return

    if lower.startswith("search "):
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        word = text[7:].strip()
        try:
            answer = f"🔎 Search: {word}\n━━━━━━━━━━━━━━━━━━━━\n\n"
            found = False
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fn in zf.namelist():
                    if is_text_file(fn):
                        for i, line in enumerate(read_file_from_zip(zf, fn).split("\n"), 1):
                            if word.lower() in line.lower():
                                answer += f"📄 {fn} line {i}: {line}\n\n"
                                found = True
            if not found:
                answer = f"❌ Nothing found for: {word}"
            if len(answer) > 4000:
                answer = answer[:4000] + "\n\n⚠️ Too many results. Shortened."
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "city":
        bot.send_message(chat_id, "🏙 Write city name.\nExample: city Almaty", reply_markup=get_zip_menu())
        return

    if lower.startswith("city "):
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        city = text[5:].strip()
        try:
            answer = f"🏙 City: {city}\n━━━━━━━━━━━━━━━━━━━━\n\n"
            found = False
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fn in zf.namelist():
                    if is_text_file(fn):
                        for line in read_file_from_zip(zf, fn).split("\n"):
                            if city.lower() in line.lower():
                                answer += f"📌 {line}\n"
                                found = True
            if not found:
                answer = f"❌ No info for city: {city}"
            if len(answer) > 4000:
                answer = answer[:4000] + "\n\n⚠️ Too many results. Shortened."
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "year":
        bot.send_message(chat_id, "📅 Write year.\nExample: year 2020", reply_markup=get_zip_menu())
        return

    if lower.startswith("year "):
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        year = text[5:].strip()
        try:
            answer = f"📅 Year: {year}\n━━━━━━━━━━━━━━━━━━━━\n\n"
            found = False
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fn in zf.namelist():
                    if is_text_file(fn):
                        for line in read_file_from_zip(zf, fn).split("\n"):
                            if year in line:
                                answer += f"📌 {line}\n"
                                found = True
            if not found:
                answer = f"❌ No info for year: {year}"
            if len(answer) > 4000:
                answer = answer[:4000] + "\n\n⚠️ Too many results. Shortened."
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "find":
        bot.send_message(chat_id, "🎯 Write city and year.\nExample: find Astana 2020", reply_markup=get_zip_menu())
        return

    if lower.startswith("find "):
        zip_path = get_user_zip(chat_id)
        if not zip_path:
            bot.send_message(chat_id, "⚠️ Please send a ZIP file first.", reply_markup=get_zip_menu())
            return
        parts = text[5:].strip().split()
        if len(parts) < 2:
            bot.send_message(chat_id, "🎯 Write city and year.\nExample: find Astana 2020", reply_markup=get_zip_menu())
            return
        year = parts[-1]
        city = " ".join(parts[:-1])
        try:
            answer = f"🎯 City: {city}  Year: {year}\n━━━━━━━━━━━━━━━━━━━━\n\n"
            found = False
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fn in zf.namelist():
                    if is_text_file(fn):
                        for line in read_file_from_zip(zf, fn).split("\n"):
                            if city.lower() in line.lower() and year in line:
                                answer += f"✅ {line}\n"
                                found = True
            if not found:
                answer = f"❌ No info for {city} and {year}"
            if len(answer) > 4000:
                answer = answer[:4000] + "\n\n⚠️ Too many results. Shortened."
            bot.send_message(chat_id, answer, reply_markup=get_zip_menu())
        except Exception as e:
            bot.send_message(chat_id, f"❌ Error: {e}", reply_markup=get_zip_menu())
        return

    if lower == "help zip":
        help_text = (
            "📦 ZIP SEARCH COMMANDS\n\n"
            "files - list files in ZIP\n"
            "summary - archive stats\n"
            "search <word> - search word in all files\n"
            "city <name> - search by city\n"
            "year <year> - search by year\n"
            "find <city> <year> - search by city + year\n\n"
            "First send a ZIP file, then use commands."
        )
        bot.send_message(chat_id, help_text, reply_markup=get_zip_menu())
        return

    bot.send_message(chat_id, "❓ Unknown command. Use the menu buttons or type /menu.", reply_markup=get_main_menu())

if __name__ == '__main__':
    print("Bot is running...")
    bot.infinity_polling()

Bot is running...


2026-05-11 17:46:08,901 (__init__.py:1134 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2026-05-11 17:46:08,905 (__init__.py:1136 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
